# INSTRUCTOR SOLUTIONS — DO NOT DISTRIBUTE

## Lesson 07: Linear Regression
## Walkthrough: Predicting Weather & Cars

**AI/ML Course | Medina County Career Center**

This is the complete solution version with all code filled in and teaching notes.

## Setup: Imports and Helper Functions

Run this cell first to load all the libraries we need.

In [ ]:
import pandas as pd                          # pandas = data tables (like Excel in Python)
import numpy as np                           # numpy = math operations on arrays
import matplotlib.pyplot as plt              # matplotlib = creates charts and graphs
import seaborn as sns                        # seaborn = makes prettier statistical charts
from sklearn.model_selection import train_test_split  # splits data into train/test
from sklearn.linear_model import LinearRegression     # linear regression algorithm
from sklearn.metrics import r2_score, mean_absolute_error  # accuracy metrics

# Set the random seed for reproducibility
np.random.seed(42)

print("All libraries loaded!")

## Sub-Lesson 07a — Correlation & Pearson r

### Instructor Note
This section introduces the concept of Pearson r correlation. Students have seen this in the Excel warmup exercises. The key points:
1. Pearson r ranges from -1 to +1
2. It measures linear relationship strength
3. R² = r² tells us the percentage of variation explained

The weather data has a VERY high correlation (r ≈ 0.98) between temp_min and temp_max because they're nearly the same measurement taken at different times of day.

### Step 1: Load the Weather Data

In [ ]:
# Load the weather data
# Relative path from ai07_Regression/ to ai06/
weatherData = pd.read_csv('medina_weather_2024.csv')

print(f"Loaded {len(weatherData)} days of weather data")
print("\nFirst 5 rows:")
print(weatherData.head())
print("\nData types:")
print(weatherData.dtypes)
print("\nBasic statistics:")
print(weatherData.describe().round(1))

### Instructor Note
The data has 365 rows (full year 2024) and 5 columns. All are numeric, which makes calculations straightforward. Check the ranges:
- temp_max: -6 to 91°F (realistic for Ohio)
- humidity: 29 to 100% (reasonable)
- precipitation: mostly 0 with occasional rain/snow events
- wind_speed: 0 to 18 mph (typical daily max winds)

### Step 2: Check for Missing Values

In [ ]:
# Check for missing values in each column
missingCounts = weatherData.isnull().sum()
print("Missing values per column:")
print(missingCounts)
print(f"\nRows before cleaning: {len(weatherData)}")

# Remove any row that has a missing value in any column
# .dropna() = "drop rows with any NaN (null) value"
weatherData = weatherData.dropna()

print(f"Rows after cleaning: {len(weatherData)}")
print(f"Rows removed: {365 - len(weatherData)}")

### Instructor Note
This dataset is clean — no missing values. This is somewhat unrealistic for real-world data. Good opportunity to mention that real data often requires more cleaning. If you're using live API data, sometimes there will be gaps.

### Step 3: Calculate Correlations (Pearson r)

In [ ]:
# Calculate Pearson r between all pairs of numeric columns
correlationMatrix = weatherData[['temp_max', 'temp_min', 'humidity', 'wind_speed', 'precipitation']].corr()

# Display as text
print("Pearson r Between All Weather Variables:\n")
print(correlationMatrix.round(3))
print("\nKey insights:")
print(f"  temp_max & temp_min: r = {correlationMatrix.loc['temp_max', 'temp_min']:.3f}  (almost perfect!)")
print(f"  temp_max & humidity: r = {correlationMatrix.loc['temp_max', 'humidity']:.3f}  (weak negative)")
print(f"  temp_max & wind_speed: r = {correlationMatrix.loc['temp_max', 'wind_speed']:.3f}  (weak negative)")

### Instructor Note
The key points to emphasize:
1. r(temp_max, temp_min) = 0.98 is suspiciously high — this is the "cheating" factor
2. r(temp_max, humidity) = -0.24 is weak but negative (higher humidity = slightly cooler)
3. r(temp_max, wind_speed) = -0.17 is very weak negative (wind has minimal effect in this dataset)
4. The diagonal is all 1.0 (a variable always correlates perfectly with itself)
5. The table is symmetric (r from A to B = r from B to A)

### Step 4: Visualize Correlations as a Heatmap

In [ ]:
# Create a pretty heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlationMatrix,
            annot=True,                      # show the r values as numbers
            cmap='coolwarm',                 # red = positive, blue = negative
            center=0,                        # center the color scale at 0
            fmt='.2f',                       # format to 2 decimal places
            vmin=-1, vmax=1)                 # scale from -1 to +1
plt.title('Pearson r Between Weather Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice the diagonal is all 1.0 (each variable correlates perfectly with itself)")
print("Notice the table is symmetric (r from A to B = r from B to A)")

### Instructor Note
The heatmap is excellent for visual understanding:
- Bright red at (temp_min, temp_max) shows the strong positive correlation
- Light blue at (temp_max, humidity) and (temp_max, wind_speed) show weak negative
- Students may ask why temp_min and wind_speed are negatively correlated (r = -0.35). Answer: Cold winter days often have more wind in Ohio. Summer warmth tends to be in calmer air masses.

### Step 5: Scatter Plot — The Strongest Relationship

In [ ]:
# Make a scatter plot: morning low (x) vs daily high (y)
plt.figure(figsize=(8, 6))
plt.scatter(weatherData['temp_min'],          # x-axis = morning low
            weatherData['temp_max'],          # y-axis = daily high
            alpha=0.5,                        # 50% transparent (so overlaps are visible)
            color='steelblue',
            s=30)                             # size of dots

plt.xlabel('Morning Low Temperature (°F)', fontsize=11)
plt.ylabel('Daily High Temperature (°F)', fontsize=11)
plt.title('Morning Low vs Daily High — Medina, OH 2024', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate correlation and R² for this pair
rValue = weatherData['temp_min'].corr(weatherData['temp_max'])
r2Value = rValue ** 2  # R² = r squared

print(f"Pearson r: {rValue:.3f}")
print(f"R-squared: {r2Value:.3f}  ({r2Value*100:.1f}% of variation explained)")

### Instructor Note
This is the moment to pause and discuss the "cheating" aspect:
- The scatter plot is almost a perfect line — unusual in real ML
- Ask students: "Why is this so tight?"
- Answer: Because temp_min and temp_max are nearly the same thing measured at different times
- This is a good learning example but a bad real-world prediction problem
- R² = 0.96 means 96% of temperature variation is explained by just the morning low
- In reality, you'd never deploy a model that predicts "today's high temperature from today's low temperature"

Transition: "This is why we also built a cars example. Let's build a model with this weather data, then compare to something more realistic."

### Step 6: The "Cheating" Conversation

In [ ]:
# Let's explicitly show why this is "cheating"
print("WHY THIS IS A 'CHEATING' PREDICTION:")
print("="*50)
print()
print("What we're doing:")
print("  Input (X):  Morning low temperature (temp_min)")
print("  Output (Y): Daily high temperature (temp_max)")
print()
print("Why it's cheating:")
print("  These are BOTH temperature measurements from the SAME DAY")
print("  They're nearly identical because:")
print("    - Same weather system")
print("    - Same sun angle")
print("    - Same atmospheric conditions")
print()
print("Real-world example:")
print(f"  If low = 25°F, high is almost always 30-40°F")
print(f"  If low = 65°F, high is almost always 75-85°F")
print(f"  This is thermodynamics, not machine learning")
print()
print("Why we use it anyway:")
print("  1. To learn the PROCESS (load, clean, explore, build, test)")
print("  2. To see what a 'too easy' prediction looks like")
print("  3. To compare against the cars example (more realistic)")
print()
print("R² comparison:")
print(f"  Weather (cheating):      R² = 0.96  (too high)")
print(f"  Cars (realistic):        R² = 0.65  (realistic)")

## Sub-Lesson 07b — Building Regression Models in Python

### Instructor Note
This section is where students learn the actual ML process. Key points:
1. Define features (X) and target (y) clearly
2. Always use train/test split
3. Train on 80%, evaluate honestly on 20%
4. Check both R² and MAE
5. Interpret coefficients in plain English

### Step 7: Build the Linear Regression Model

In [ ]:
# Define the features (X) and target (y)
# X = the inputs (what we use to predict)
# y = the output (what we're predicting)
featureColumns = ['temp_min', 'humidity', 'wind_speed']
X = weatherData[featureColumns]               # 3 input columns
y = weatherData['temp_max']                   # the target column

print(f"Features shape: {X.shape}  (rows, columns)")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns:")
for col in featureColumns:
    print(f"  - {col}")

In [ ]:
# Split the data: 80% for training, 20% for testing
# random_state=42 ensures we get the same split every time (reproducible)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)} days")
print(f"Test set: {len(X_test)} days (the model will never see these)")
print(f"\nTotal: {len(X_train) + len(X_test)} days")
print(f"\nSplit ratio:")
print(f"  Training: {len(X_train)/len(X)*100:.1f}%")
print(f"  Testing:  {len(X_test)/len(X)*100:.1f}%")

### Instructor Note
The 80/20 split is standard in ML. Explain why:
- 80% gives the model enough data to learn patterns
- 20% is enough to get a stable evaluation (73 days in this case)
- random_state=42 is arbitrary but reproducible — useful for debugging
- Always split BEFORE looking at test data — otherwise you're cheating

In [ ]:
# Create an empty Linear Regression model
model = LinearRegression()

# .fit() trains the model — it finds the best coefficients
# The algorithm: minimize the sum of squared residuals
model.fit(X_train, y_train)

print("✓ Model trained successfully!")
print(f"  Coefficients found: {len(model.coef_)}")
print(f"  Intercept found: 1")

### Instructor Note
What's happening under the hood:
- LinearRegression uses "ordinary least squares" (OLS) algorithm
- It finds coefficients that minimize: sum((actual - predicted)²)
- This is a closed-form solution (fast) unlike neural networks (iterative)
- sklearn handles all the math; we just call .fit()

### Step 8: Evaluate the Model

In [ ]:
# Use the trained model to make predictions on the test set
predictions = model.predict(X_test)          # array of predictions

# Calculate accuracy metrics
r2 = r2_score(y_test, predictions)           # R² = % of variation explained
mae = mean_absolute_error(y_test, predictions)  # MAE = average error in degrees

print("MODEL PERFORMANCE ON TEST DATA:")
print(f"  R² Score:  {r2:.4f}  ({r2*100:.1f}%)")
print(f"  MAE:       {mae:.2f} degrees F")
print(f"\nInterpretation:")
print(f"  Our model explains {r2*100:.1f}% of the variation in daily highs.")
print(f"  On average, our predictions are {mae:.1f} degrees off.")
print(f"\nReminder: This R² is high because temp_min and temp_max are nearly")
print(f"the same measurement. The cars model will be more realistic.")

### Instructor Note
Analyze the results:
- R² ≈ 0.92 is very high but expected given the relationship
- MAE ≈ 2.5°F means typical prediction error is ±2.5 degrees
- This is still amazing accuracy for weather, but remember: we're "cheating"
- Contrast: predicting next week's weather is R² ≈ 0.3 because many factors change
- But predicting today's high from today's low is R² ≈ 0.92 because they're the same day

### Step 9: Visualize Predictions vs Actual

In [ ]:
# Create a scatter plot showing actual vs predicted temperatures
plt.figure(figsize=(8, 6))

# Draw the "perfect prediction" diagonal line
minTemp = y_test.min()
maxTemp = y_test.max()
plt.plot([minTemp, maxTemp],                  # x range (actual temps)
         [minTemp, maxTemp],                  # y range (perfect predictions)
         'r--', linewidth=2, label='Perfect predictions')

# Plot our actual predictions
plt.scatter(y_test, predictions,              # x = actual, y = predicted
            alpha=0.5, color='steelblue', s=40,
            label='Our predictions')

plt.xlabel('Actual Temperature (°F)', fontsize=11)
plt.ylabel('Predicted Temperature (°F)', fontsize=11)
plt.title('How Close Are Our Predictions?', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Dots close to the red line = good predictions")
print("Dots far from the line = the model was off for that day")

### Instructor Note
The plot shows:
- Most points are very close to the red line
- No obvious bias (points aren't systematically above or below)
- A few outliers (dots far from the line) — these are unusual weather days
- This visual confirmation matches the high R² score
- If points were scattered randomly, that would indicate a bad model

### Step 10: Understand the Coefficients

In [ ]:
# Create a table of coefficients
coefficientTable = pd.DataFrame({
    'Feature': featureColumns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)  # sort by magnitude

print("WHAT THE MODEL LEARNED:\n")
print("The formula:")
print(f"  Predicted temp_max = ({model.coef_[0]:+.3f} × temp_min)")
print(f"                     + ({model.coef_[1]:+.3f} × humidity)")
print(f"                     + ({model.coef_[2]:+.3f} × wind_speed)")
print(f"                     + {model.intercept_:.2f}  (intercept)")

print("\nInterpretation:")
print(f"  Temp_min:    {model.coef_[0]:+.3f}  → Each 1° warmer morning ≈ {model.coef_[0]:.2f}° warmer high")
print(f"  Humidity:    {model.coef_[1]:+.3f}  → Each 1% more humidity ≈ {model.coef_[1]:.3f}° cooler high")
print(f"  Wind speed:  {model.coef_[2]:+.3f}  → Each 1 mph more wind ≈ {model.coef_[2]:.3f}° cooler high")

### Instructor Note
Coefficients tell us:
- temp_min coefficient ≈ 1.03: Almost 1:1 relationship (makes sense — same day)
- humidity coefficient ≈ -0.26: Higher humidity = slightly cooler (typical meteorology)
- wind_speed coefficient ≈ -0.16: Wind has slight cooling effect (evaporative)
- Intercept ≈ 9: This is the baseline prediction if all features were 0 (nonsensical scenario)

Key teaching point: Coefficients only make sense for the range of data you trained on. You wouldn't use this to predict a day with temp_min = -50 or humidity = 200%.

In [ ]:
# Bar chart showing which features have the biggest impact
plt.figure(figsize=(8, 4))

# Color: green for positive, red for negative
colors = ['green' if c > 0 else 'red' for c in coefficientTable['Coefficient']]

plt.barh(coefficientTable['Feature'],         # y-axis = feature names
         coefficientTable['Coefficient'],     # x-axis = coefficient values
         color=colors)
plt.xlabel('Effect on Predicted Temperature', fontsize=11)
plt.title('Which Features Matter Most?', fontsize=12, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.8)  # vertical line at 0
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("Green bars = positive effect (pushes temperature UP)")
print("Red bars = negative effect (pushes temperature DOWN)")
print(f"\nObservation: temp_min dominates. That's why this prediction is so easy.")

### Instructor Note
The bar chart makes it obvious:
- temp_min has a huge positive effect (1.03 coefficient)
- humidity and wind_speed have tiny negative effects (< 0.3)
- This explains the high R² — one feature (temp_min) explains most variation
- In the cars example, features will be more balanced, making the problem harder

### Step 11: Make a Prediction

In [ ]:
# Create a hypothetical day's conditions
# Column names MUST match what the model was trained on
hypotheticalDay = pd.DataFrame({
    'temp_min': [45],                        # morning low = 45 F
    'humidity': [65],                        # humidity = 65%
    'wind_speed': [10]                       # wind speed = 10 mph
})

# Make the prediction
predictedTemp = model.predict(hypotheticalDay)[0]  # [0] grabs the single number

print(f"Given conditions:")
print(f"  Morning low = 45°F")
print(f"  Humidity = 65%")
print(f"  Wind speed = 10 mph")
print(f"\nModel's prediction: {predictedTemp:.1f}°F")

print(f"\nLet's verify the math manually:")
manualCalc = (model.coef_[0] * 45 +
              model.coef_[1] * 65 +
              model.coef_[2] * 10 +
              model.intercept_)
print(f"  ({model.coef_[0]:.3f} × 45) + ({model.coef_[1]:.3f} × 65) + ({model.coef_[2]:.3f} × 10) + {model.intercept_:.2f}")
print(f"  = {manualCalc:.1f}°F  ✓ (same answer!)")

### Instructor Note
Verifying the math:
- Shows students how the formula works
- Transparency is important — they can check the math themselves
- Try different scenarios to help them build intuition:
  - Higher temp_min → higher prediction (obvious)
  - Higher humidity → slightly lower (learned from data)
  - Higher wind → slightly lower (learned from data)

## Summary: What We Learned

**Sub-Lesson 07a — Correlation & Pearson r:**
- Pearson r ranges from -1 to +1 and measures relationship strength
- R² = r × r tells us the percentage of variation explained
- Correlation heatmaps show relationships at a glance

**Sub-Lesson 07b — Building Regression Models:**
- Linear regression finds the best-fit line through data
- Train/test split (80/20) ensures honest evaluation
- R² and MAE tell us model quality
- Coefficients show how each feature affects predictions
- The weather example was "cheating" (temp_min ≈ temp_max on same day)

**Key takeaway for students:**
- This process applies to ANY regression problem
- The cars example shows how it works with more realistic data
- Other models (random forests, neural networks) use the same train/test approach